# Plantilla del proyecto - Clase 5 (guia §5)

Grupos de 3, un dominio cada grupo, un solo `main.py` que recorre la
cadena: ADQUISICION -> PREPROCESAMIENTO -> SEGMENTACION/DETECCION ->
EXTRACCION -> ML/DL -> ANALISIS -> VISUALIZACION -> INTERACCION.

**La regla T2:** antes de llenar nada, este esqueleto tiene que ejecutar de
principio a fin con datos minimos. Las celdas de codigo de abajo ya
ejecutan: son la firma de cada etapa (que recibe, que devuelve) y un main
que las encadena. Al final de cada funcion hay un marcador `completa_aqui`
con instrucciones; el codigo que las sustituya tiene que cumplir el
contrato escrito en el docstring.

## Objetivos

1. Encadenar las ocho etapas en un programa unico con datos que no entrenaron.
2. Declarar el contrato de cada etapa en su firma.
3. Medir el tiempo por etapa y el coste FN/FP del dominio.
4. Cerrar con la cifra que decide, sostenida por una figura.
5. Entregar `main.py`, `figuras/`, `resultados.txt` y `analisis.md` (guia §5.1).

## Preparacion

El dominio y su fuente se eligen en T1 (guia §7). Este esqueleto usa una
fuente sintetica minima para que T2 pueda correr en cualquier aula; el
grupo la sustituye por la suya (camara, motor o lote real).

In [ ]:
import sys
import time
from pathlib import Path

CURSO = Path.cwd()
for candidata in (CURSO, CURSO.parent, CURSO / 'computer-vision-course', CURSO.parent / 'computer-vision-course'):
    if (candidata / 'cvcourse').exists():
        CURSO = candidata
        break
else:
    raise RuntimeError(f'no encuentro el curso desde {CURSO}')
if str(CURSO) not in sys.path:
    sys.path.insert(0, str(CURSO))

import numpy as np

from cvcourse import features, synthetic

SEMILLA = 42   # fija, siempre
print('esqueleto listo en', CURSO)

## El experimento

### T1 - Dominio y fuente

Elegir dominio de la tabla de la guia §7 (o proponer uno propio con la
misma arquitectura) y justificar la fuente: por que los datos de ese
dominio entran por esa puerta. Escribirlo aqui, con una frase que se
sostenga sola.

### T2 - El esqueleto de 8 etapas

Las funciones de abajo son los contratos. Cada docstring dice QUE recibe y
QUE devuelve la etapa; el cuerpo minimo es un passthrough para que el
esqueleto ejecute con datos minimos ANTES de llenar nada. Al implementar,
cada etapa toma su pieza de las clases 1-4 y cita la ruta concreta.

In [ ]:
def etapa_1_adquisicion(origen):
    """Recibe: una fuente ('camara', 'motor', 'sintetica', 'lote').
    Devuelve: imagenes (list[np.ndarray], RGB o gris uint8).
    Pieza de clase anterior: class01_acquisition/ (o el origen propio)."""
    rng = np.random.default_rng(SEMILLA)
    imagenes = [rng.integers(0, 256, (64, 64), dtype=np.uint8) for _ in range(5)]
    # completa_aqui: sustituir por tu fuente real (camara, motor, lote).
    return imagenes

def etapa_2_preprocesamiento(imagenes):
    """Recibe: imagenes crudas. Devuelve: imagenes limpias (misma forma)."""
    # completa_aqui: material de la Clase 2 (filtrado, realce).
    return list(imagenes)

def etapa_3_segmentacion(imagenes):
    """Recibe: imagenes limpias. Devuelve: mascaras (bool, una por imagen)."""
    # completa_aqui: material de la Clase 3 (Otsu, morfologia, watershed).
    return [img > 127 for img in imagenes]

def etapa_4_extraccion(mascaras):
    """Recibe: mascaras. Devuelve: filas de caracteristicas (una por region)."""
    filas = []
    for mascara in mascaras:
        filas.extend(features.caracteristicas_de_mascara(mascara, etiqueta_de_clase='OK'))
    # completa_aqui: etiquetar con tu clase real, no con 'OK'.
    return filas

def etapa_5_ml(filas):
    """Recibe: filas. Devuelve: (modelo entrenado, predicciones con proba)."""
    X, y, _ = features.a_matriz(filas)
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.model_selection import train_test_split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.3, random_state=SEMILLA, stratify=y)
    knn = KNeighborsClassifier().fit(X_tr, y_tr)
    print('esperanza minima: acc', round(knn.score(X_te, y_te), 3),
          'sobre', len(X_te), 'piezas que no entrenaron')
    # completa_aqui: cinco modelos y tabla, como la Clase 4.
    return knn, knn.predict(X_te)

def etapa_6_analisis(predicciones):
    """Recibe: predicciones. Devuelve: decision con coste (dict)."""
    fn = int((predicciones != 'OK').sum())  # completa_aqui: FN y FP de verdad
    return {'fn': fn, 'fp': 0, 'coste': fn * 20.0}

def etapa_7_visualizacion(datos):
    """Recibe: lo que haya que pintar. Devuelve: rutas de figuras."""
    # completa_aqui: al menos una figura por etapa con trabajo que mostrar.
    return ['figuras/ (pendiente en T7)']

def etapa_8_interaccion(reporte):
    """Recibe: la decision. Devuelve: accion o reporte reproducible"""
    print('reporte reproducible:', reporte)
    # completa_aqui: tu accion (orden de agarre, resumen de turno, alerta...).
    return reporte

In [ ]:
# El encadenado: solo esto cambia de orden si el contrato lo pide.
def main():
    etapas = []
    t0 = time.perf_counter()
    imagenes = etapa_1_adquisicion('sintetica')
    etapas.append(('1. ADQUISICION', time.perf_counter() - t0))
    t0 = time.perf_counter()
    limpias = etapa_2_preprocesamiento(imagenes)
    etapas.append(('2. PREPROCESAMIENTO', time.perf_counter() - t0))
    t0 = time.perf_counter()
    mascaras = etapa_3_segmentacion(limpias)
    etapas.append(('3. SEGMENTACION', time.perf_counter() - t0))
    t0 = time.perf_counter()
    filas = etapa_4_extraccion(mascaras)
    etapas.append(('4. EXTRACCION', time.perf_counter() - t0))
    t0 = time.perf_counter()
    modelo, pred = etapa_5_ml(filas)
    etapas.append(('5. ML/DL', time.perf_counter() - t0))
    t0 = time.perf_counter()
    reporte = etapa_6_analisis(pred)
    etapas.append(('6. ANALISIS', time.perf_counter() - t0))
    t0 = time.perf_counter()
    figuras = etapa_7_visualizacion({'filas': filas, 'pred': pred})
    etapas.append(('7. VISUALIZACION', time.perf_counter() - t0))
    t0 = time.perf_counter()
    etapa_8_interaccion(reporte)
    etapas.append(('8. INTERACCION', time.perf_counter() - t0))
    print()
    print('Tiempo por etapa (medido en esta misma ejecucion):')
    for nombre, segundos in etapas:
        print(f'    {nombre:20s} {segundos * 1000:8.2f} ms')
    return 0

raise SystemExit(main())  # T2: el esqueleto ejecuta antes de llenar nada.

### T3 a T8 - Como llenar cada etapa

- **T3 (adquisicion + preprocesamiento):** material de Clases 1 y 2; medir el
  tamano de la imagen que entra y el tiempo de la etapa.
- **T4 (segmentacion + extraccion):** material de Clase 3; medir cuantas
  regiones salen y cuantas filas entran a la tabla.
- **T5 (modelo):** material de Clase 4; particion honesta con semilla fija,
  linea base al lado y eleccion por las celdas de la matriz.
- **T6 (analisis):** la decision con su coste; la cifra que decide se reporta
  con su coste, nunca pelada.
- **T7 (visualizacion):** al menos una figura por etapa con trabajo que
  mostrar, y la figura final del sistema.
- **T8 (interaccion):** el sistema termina en una accion o un reporte
  reproducible; `resultados.txt` coincide con lo que imprime `main.py`.

## Reto (opcional, guia §5.5)

Insertar una variacion en UNA etapa y medir la consecuencia en la cifra
final: subir el ruido de la adquisicion, cambiar el umbral de Otsu por uno
fijo, quitar la morfologia, o cambiar el modelo por el peor de la tabla de
la Clase 4. El sistema debe seguir ejecutando y el analisis debe reportar
la diferencia medida.

## Preguntas de análisis

Responder en `analisis.md` (maximo 2 paginas), cada una con una cifra o una
figura detras (guia §5.4):

1. ¿Cual es la cifra que decide en su sistema y como se midio? ¿Cuanto
   cuesta equivocarse en cada direccion (la pareja FN/FP de su dominio)?
2. ¿Que etapa es la mas lenta y cuanto pesa en el tiempo total? ¿Que pasaria
   en su dominio si esa etapa costara el doble?
3. ¿Que etapa fue la mas dificil de integrar y que contrato tuvieron que
   aclarar?
4. ¿Que pasaria si la adquisicion cambiara? ¿Que etapa absorbe el cambio y
   cual se rompe primero?
5. ¿Que pieza de las clases 1-4 usa cada etapa? Cite la ruta concreta.

## Conclusiones

Se completan al cerrar el proyecto: la figura y la cifra de la presentacion
de 5 minutos, y la puesta en comun de que etapa fue la mas cara de
integrar.

## Bibliografía

- Guia: `docs/clase05_guia.md` (arquitectura obligatoria y rubrica).
- Referencias por dominio: `examples/class05_integration/`.
- Material de clases 1-4: `examples/class0{1..4}_*/`.
- Solucion de referencia: `solutions/clase05_solucion.py`.